# S4003 — Synthèse unifiée : EWC INT8 sur MCU (figures de l'article)

**Source unique** des chiffres et figures de l'article *« EWC INT8 sur microcontrôleur :
effondrement de la PTQ legacy et récupération par kernel calibré »*.

Toutes les valeurs proviennent d'un `json.load` — **aucun chiffre en dur** ni ici ni dans le
`.tex`. Trois campagnes rechargées :

| Source | Contenu |
|--------|---------|
| `exp_S36_summary.json` + `exp_S36_parity_*.json` | PC↔board FP32 + INT8 legacy (frozen/online) |
| `exp_S39_ablation/{ds}.json` | échelle d'ablation `legacy_c → … → q15` (émulé PC) |
| `exp_S39_quant_sweep/ewc_{ds}.json` | schémas + RAM + proxy latence (émulé PC) |
| `exp_S40_board_v2/*.json` | récupération board du kernel v2 — **`"à mesurer"` si absent** |

**Convention visuelle** : *plein* = mesuré board (matériel), *hachuré/gris* = émulé PC.
Cellules board v2 absentes → masquées / annotées `"à mesurer"` (règle « aucun chiffre inventé »).
Le notebook s'exécute (`nbconvert --execute`) **même sans** `exp_S40_board_v2/` (dégradation gracieuse).

In [1]:
import json, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Détection de la racine du dépôt (remonte jusqu'à experiments/) ──
ROOT = Path.cwd()
while not (ROOT / "experiments").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.evaluation.plots import save_figure  # style homogène projet

EXP = ROOT / "experiments"
FIGS = ROOT / "docs" / "figures" / "sprint40_article"
FIGS.mkdir(parents=True, exist_ok=True)
DATASETS = ["pronostia", "monitoring"]
NA = "à mesurer"                      # règle « aucun chiffre inventé »
LATENCY_BUDGET_US = 100_000.0         # Gap 2 : 100 ms

# Style : plein = mesuré board, hachuré = émulé PC
BOARD_KW = dict(alpha=0.95, edgecolor="black", linewidth=0.8)
EMU_KW   = dict(alpha=0.55, edgecolor="black", linewidth=0.8, hatch="//")
C_FP32, C_LEGACY, C_PC, C_Q15 = "#2c7fb8", "#d95f02", "#1b9e77", "#7570b3"

def load(rel):
    p = EXP / rel
    return json.loads(p.read_text()) if p.exists() else None

# ── Sources ──
S36 = load("exp_S36_summary.json")["results"]
PARITY = {(cond, proto, ds): load(f"exp_S36_parity_{cond}_{proto}_{ds}.json")
          for cond in ["5feat"] for proto in ["frozen", "online"] for ds in DATASETS}
ABL = {ds: load(f"exp_S39_ablation/{ds}.json") for ds in DATASETS}
QSW = {ds: load(f"exp_S39_quant_sweep/ewc_{ds}.json") for ds in DATASETS}
S40 = {(sch, ds, pr): load(f"exp_S40_board_v2/results_{sch}_{ds}_{pr}.json")
       for sch in ["per_channel", "q15", "int8_legacy"] for ds in DATASETS
       for pr in ["frozen", "online"]}
HAS_S40 = any(v is not None for v in S40.values())
print("Sources chargées. exp_S40_board_v2 présent :", HAS_S40)
provenance = []  # (grandeur, valeur, source_json, statut) → tableau récap final
def note(label, value, src, status):
    provenance.append({"grandeur": label, "valeur": value, "source_json": src, "statut": status})

Sources chargées. exp_S40_board_v2 présent : True


## Figure 1 — Parité FP32 PC ↔ board

Le portage FP32 est **exact en frozen** (poids gelés → `parity_rate = 1.000`) et **approché en
online** (float32 board ≠ float64 PC → 0.96–0.99). Source : `exp_S36_parity_5feat_*.json`.

In [2]:
fig, ax = plt.subplots(figsize=(7, 4.2))
x = np.arange(len(DATASETS)); w = 0.36
froz = [PARITY[("5feat","frozen",ds)]["parity_rate"] if PARITY[("5feat","frozen",ds)] else np.nan for ds in DATASETS]
onl  = [PARITY[("5feat","online",ds)]["parity_rate"] if PARITY[("5feat","online",ds)] else np.nan for ds in DATASETS]
ax.bar(x - w/2, froz, w, label="frozen (exact)", color=C_FP32, **BOARD_KW)
ax.bar(x + w/2, onl,  w, label="online (approché)", color=C_PC, **BOARD_KW)
for xi, (f, o) in enumerate(zip(froz, onl)):
    if not np.isnan(f): ax.text(xi - w/2, f + 0.005, f"{f:.3f}", ha="center", va="bottom", fontsize=9)
    if not np.isnan(o): ax.text(xi + w/2, o + 0.005, f"{o:.3f}", ha="center", va="bottom", fontsize=9)
    note(f"parité FP32 frozen {DATASETS[xi]}", f, "exp_S36_parity_5feat_frozen_%s.json" % DATASETS[xi], "mesuré board")
    note(f"parité FP32 online {DATASETS[xi]}", o, "exp_S36_parity_5feat_online_%s.json" % DATASETS[xi], "mesuré board")
ax.set_xticks(x); ax.set_xticklabels(DATASETS); ax.set_ylim(0.90, 1.01)
ax.set_ylabel("taux de parité board↔PC"); ax.set_title("Parité FP32 PC ↔ NUCLEO-F439ZI (condition 5feat)")
ax.legend(); ax.grid(axis="y", alpha=0.3)
save_figure(fig, FIGS / "fig1_parity_fp32_pc_board.png")

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint40_article/fig1_parity_fp32_pc_board.png


## Figure 2 — Latences (Gap 2)

Inférence seule (board frozen) vs inférence + MAJ CL (board online), comparées à la latence PC.
Échelle log µs, ligne budget 100 ms (Gap 2). Barres pleines = board ; PC = référence.
Si la campagne v2 est présente, ses latences board (per_channel/q15) sont ajoutées.

In [3]:
fig, ax = plt.subplots(figsize=(8.5, 4.6))
labels, vals, colors, kws = [], [], [], []
for ds in DATASETS:
    c = S36[ds]["5feat"]
    pc_us = c["pc"]["inference_latency_ms"] * 1000.0
    bf = c["board_frozen"]["latency_us_p50"]; bo = c["board_online"]["latency_us_p50"]
    for lab, v, col, kw, src, st in [
        (f"{ds}\nPC inf", pc_us, "#999999", EMU_KW, "exp_S36_summary.json", "mesuré PC"),
        (f"{ds}\nboard inf", bf, C_FP32, BOARD_KW, "exp_S36_summary.json", "mesuré board"),
        (f"{ds}\nboard inf+MAJ", bo, C_LEGACY, BOARD_KW, "exp_S36_summary.json", "mesuré board")]:
        labels.append(lab); vals.append(v); colors.append(col); kws.append(kw)
        note(lab.replace(chr(10)," "), v, src, st)
    if HAS_S40:
        for sch, col in [("per_channel", C_PC), ("q15", C_Q15)]:
            r = S40[(sch, ds, "frozen")]
            if r and r.get("latency_us_p50") is not None:
                labels.append(f"{ds}\nv2 {sch}"); vals.append(r["latency_us_p50"]); colors.append(col); kws.append(BOARD_KW)
                note(f"latence board v2 {sch} {ds}", r["latency_us_p50"], f"exp_S40_board_v2/results_{sch}_{ds}_frozen.json", "mesuré board")
xs = np.arange(len(labels))
for xi, (v, col, kw) in enumerate(zip(vals, colors, kws)):
    ax.bar(xi, v, 0.8, color=col, **kw)
    ax.text(xi, v * 1.05, f"{v:.0f}", ha="center", va="bottom", fontsize=8)
ax.axhline(LATENCY_BUDGET_US, color="red", ls="--", lw=1.3, label="budget Gap 2 (100 ms)")
ax.set_yscale("log"); ax.set_xticks(xs); ax.set_xticklabels(labels, fontsize=7.5)
ax.set_ylabel("latence P50 (µs, log)"); ax.set_title("Latences EWC — PC vs board (Gap 2)")
ax.legend(); ax.grid(axis="y", alpha=0.3)
if not HAS_S40:
    ax.text(0.99, 0.02, "v2 board : à mesurer", transform=ax.transAxes, ha="right", fontsize=8, style="italic", color="gray")
save_figure(fig, FIGS / "fig2_latency_gap2.png")

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint40_article/fig2_latency_gap2.png


## Figure 3 — Échelle d'ablation INT8 (émulé PC)

`legacy_c → fix_acc32 → per_tensor_calib → per_channel → q15` : le saut décisif (+0.88 F1)
apparaît au **scale calibré** (`per_tensor_calib`), isolant la cause racine (le `1/128` figé,
pas l'overflow int16). Source : `exp_S39_ablation/{ds}.json` (hachuré = émulé).

In [4]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(12, 4.4), sharey=True)
for ax, ds in zip(np.atleast_1d(axes), DATASETS):
    a = ABL[ds]
    steps = [s["scheme"] for s in a["ladder"]]; f1s = [s["f1"] for s in a["ladder"]]
    ax.bar(range(len(steps)), f1s, color=C_PC, **EMU_KW)
    ax.axhline(a["f1_fp32"], color="black", ls=":", lw=1.4, label=f"F1 FP32 = {a['f1_fp32']:.3f}")
    # annote le plus gros saut (delta_prev max)
    deltas = [(i, s["delta_prev"]) for i, s in enumerate(a["ladder"]) if s.get("delta_prev") is not None]
    if deltas:
        i, dmax = max(deltas, key=lambda t: t[1])
        ax.annotate(f"+{dmax:.2f}", xy=(i, f1s[i]), xytext=(i, f1s[i] + 0.12),
                    ha="center", fontsize=11, fontweight="bold", color="darkred",
                    arrowprops=dict(arrowstyle="->", color="darkred"))
        note(f"saut F1 ablation {ds} ({steps[i]})", dmax, f"exp_S39_ablation/{ds}.json", "émulé PC")
    for i, v in enumerate(f1s):
        ax.text(i, v + 0.01, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(range(len(steps))); ax.set_xticklabels(steps, rotation=30, ha="right", fontsize=8)
    ax.set_title(f"{ds}"); ax.grid(axis="y", alpha=0.3); ax.legend(fontsize=8)
np.atleast_1d(axes)[0].set_ylabel("F1_faulty (émulé PC)")
fig.suptitle("Échelle d'ablation de la quantification INT8 (EWC, 5feat)")
fig.tight_layout()
save_figure(fig, FIGS / "fig3_ablation_ladder.png")

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint40_article/fig3_ablation_ladder.png


## Figure 4 — INT8 vs FP32 sur board : effondrement puis récupération

**Avant** (legacy, board S36) : effondrement F1. **Après** (kernel v2) : récupération —
émulateur PC (hachuré, `exp_S39_quant_sweep`) et **board** (plein, `exp_S40_board_v2`, ou
`"à mesurer"` si la campagne n'a pas tourné). Sous-panneau : accord INT8↔FP32 + RAM ÷4/÷2 (Gap 3).

In [5]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(12, 4.8), sharey=True)
for ax, ds in zip(np.atleast_1d(axes), DATASETS):
    c = S36[ds]["5feat"]; q = QSW[ds]["schemes"]
    bars = []  # (label, value, color, kw, src, status)
    bars.append(("FP32\nboard", c["board_frozen"]["f1_faulty"], C_FP32, BOARD_KW, "exp_S36_summary.json", "mesuré board"))
    bars.append(("legacy\nboard", c["board_frozen_int8"]["f1_faulty"], C_LEGACY, BOARD_KW, "exp_S36_summary.json", "mesuré board"))
    bars.append(("per_ch\némulé", q["int8_perchannel"]["metric"], C_PC, EMU_KW, f"exp_S39_quant_sweep/ewc_{ds}.json", "émulé PC"))
    bars.append(("q15\némulé", q["q15"]["metric"], C_Q15, EMU_KW, f"exp_S39_quant_sweep/ewc_{ds}.json", "émulé PC"))
    for sch, col, lab in [("per_channel", C_PC, "per_ch\nboard"), ("q15", C_Q15, "q15\nboard")]:
        r = S40[(sch, ds, "frozen")]
        val = r["f1_faulty"] if (r and r.get("f1_faulty") is not None) else np.nan
        bars.append((lab, val, col, BOARD_KW, f"exp_S40_board_v2/results_{sch}_{ds}_frozen.json",
                     "mesuré board" if r else NA))
    for i, (lab, v, col, kw, src, st) in enumerate(bars):
        if np.isnan(v):
            ax.text(i, 0.02, NA, rotation=90, ha="center", va="bottom", fontsize=8, style="italic", color="gray")
        else:
            ax.bar(i, v, 0.8, color=col, **kw)
            ax.text(i, v + 0.01, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
        note(f"F1 {lab.replace(chr(10),' ')} {ds}", None if np.isnan(v) else v, src, st)
    ax.set_xticks(range(len(bars))); ax.set_xticklabels([b[0] for b in bars], fontsize=8)
    ax.set_title(ds); ax.grid(axis="y", alpha=0.3); ax.set_ylim(0, 1.05)
np.atleast_1d(axes)[0].set_ylabel("F1_faulty")
handles = [mpatches.Patch(facecolor="gray", **{k: v for k, v in BOARD_KW.items() if k != "alpha"}, label="mesuré board"),
           mpatches.Patch(facecolor="gray", hatch="//", alpha=0.55, edgecolor="black", label="émulé PC")]
fig.legend(handles=handles, loc="upper right", fontsize=9)
fig.suptitle("EWC INT8 board : effondrement legacy → récupération kernel v2")
fig.tight_layout()
save_figure(fig, FIGS / "fig4_int8_recovery_board.png")

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint40_article/fig4_int8_recovery_board.png


## Figure 5 — Pareto RAM × F1 × latence

Nuage {FP32, INT8 legacy, per_channel, q15} : RAM des poids (x), F1/métrique (y), proxy latence
(taille). Émulé PC (`exp_S39_quant_sweep`) en hachuré ; points board v2 (plein) si présents.
Illustre le compromis Gap 3 : per_channel/q15 récupèrent la F1 en divisant la RAM (÷4 / ÷2).

In [6]:
fig, ax = plt.subplots(figsize=(8, 5.2))
scheme_style = {"fp32": (C_FP32, "FP32"), "int8_legacy": (C_LEGACY, "INT8 legacy"),
                "int8_perchannel": (C_PC, "per_channel"), "q15": (C_Q15, "q15")}
for ds, marker in zip(DATASETS, ["o", "s"]):
    q = QSW[ds]["schemes"]
    for sch, (col, lab) in scheme_style.items():
        s = q[sch]
        ram = s["ram_weights_bytes"]; f1 = s["metric"]; latr = s.get("lat_proxy_rel", 1.0)
        ax.scatter(ram, f1, s=120 + 260 * latr, c=col, marker=marker, alpha=0.55,
                   edgecolors="black", linewidths=0.8, hatch="//")
        ax.annotate(f"{lab}", (ram, f1), fontsize=7, xytext=(4, 4), textcoords="offset points")
        note(f"RAM poids {lab} {ds}", ram, f"exp_S39_quant_sweep/ewc_{ds}.json", "émulé PC")
    if HAS_S40:
        for sch in ["per_channel", "q15"]:
            r = S40[(sch, ds, "frozen")]
            if r and r.get("f1_faulty") is not None:
                ax.scatter(r["ram_weights_quant_bytes"], r["f1_faulty"], s=200,
                           c=scheme_style["int8_perchannel" if sch=="per_channel" else "q15"][0],
                           marker=marker, edgecolors="black", linewidths=1.4, label=None)
                note(f"RAM board v2 {sch} {ds}", r["ram_weights_quant_bytes"],
                     f"exp_S40_board_v2/results_{sch}_{ds}_frozen.json", "mesuré board")
ax.set_xlabel("RAM poids (octets)"); ax.set_ylabel("F1 / métrique")
ax.set_title("Pareto RAM × F1 × latence (taille ∝ proxy latence)")
ax.grid(alpha=0.3)
mk = [plt.Line2D([0],[0], marker=m, color="gray", ls="", label=ds) for ds, m in zip(DATASETS, ["o","s"])]
ax.legend(handles=mk, title="dataset (hachuré=émulé, plein=board)", fontsize=8)
save_figure(fig, FIGS / "fig5_pareto_ram_f1_latency.png")

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint40_article/fig5_pareto_ram_f1_latency.png


## Tableau récapitulatif — source unique du `.tex`

Chaque grandeur utilisée dans l'article, sa valeur, son fichier JSON d'origine et son statut
(`mesuré board` / `mesuré PC` / `émulé PC` / `à mesurer`). Les valeurs `à mesurer` correspondent
aux cellules board v2 tant que `exp_S40_board_v2/` n'a pas été produit (règle « aucun chiffre inventé »).

In [7]:
import pandas as pd
df = pd.DataFrame(provenance)
df["valeur"] = df["valeur"].apply(lambda v: NA if v is None else (round(v, 4) if isinstance(v, float) else v))
print(f"{len(df)} grandeurs — statuts : {df['statut'].value_counts().to_dict()}")
(FIGS / "provenance_table.csv").write_text(df.to_csv(index=False))
df

34 grandeurs — statuts : {'mesuré board': 15, 'émulé PC': 14, 'à mesurer': 3, 'mesuré PC': 2}


,grandeur,valeur,source_json,statut
0,parité FP32 frozen pronostia,1.0000,exp_S36_parity_5feat_frozen_pronostia.json,mesuré board
1,parité FP32 online pronostia,0.9754,exp_S36_parity_5feat_online_pronostia.json,mesuré board
2,parité FP32 frozen monitoring,1.0000,exp_S36_parity_5feat_frozen_monitoring.json,mesuré board
3,parité FP32 online monitoring,0.9887,exp_S36_parity_5feat_online_monitoring.json,mesuré board
4,pronostia PC inf,33.7509,exp_S36_summary.json,mesuré PC
5,pronostia board inf,50.0000,exp_S36_summary.json,mesuré board
6,pronostia board inf+MAJ,251.0000,exp_S36_summary.json,mesuré board
7,latence board v2 per_channel pronostia,68.0000,exp_S40_board_v2/results_per_channel_pronostia...,mesuré board
8,monitoring PC inf,34.6429,exp_S36_summary.json,mesuré PC
9,monitoring board inf,48.0000,exp_S36_summary.json,mesuré board
